<a href="https://colab.research.google.com/github/mrishab/Semantic-Communication-Paper/blob/main/SemanticCommunication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.8/570.8 kB 9.7 MB/s  0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.3.0
    Uninstalling sentence-transformers-5.3.0:
      Successfully uninstalled sentence-transformers-5.3.0


In [2]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors


# Setting up Data, Embeddings, and the Rebuilder Model

For creating embeddings, I am using existing pretrained model `all-MiniLM-L6-v2` to convert text into 384-dimensional embeddings.

Then I'll use an `MLPRegressor` to learn how to map the embeddings back into the original text.

Defining Sample Sentences

In [3]:
sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "Data science is an interdisciplinary field.",
    "Artificial intelligence is transforming the world.",
    "I love working with python and pandas.",
    "Jupyter notebooks are great for prototyping.",
    "Natural language processing allows computers to understand text.",
    "Machine learning models require clean data for training."
]

Encoding the Sentences using pretrained transformer

In [4]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
X = embedder.encode(sentences)

X

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


array([[ 0.04393354,  0.05893438,  0.0481784 , ...,  0.05216282,
         0.05610644,  0.10206395],
       [ 0.00783482, -0.02188822, -0.0135582 , ..., -0.0132392 ,
        -0.02153722, -0.00871551],
       [ 0.03872418, -0.00110565,  0.0827162 , ..., -0.02902938,
         0.04854377, -0.03839871],
       ...,
       [-0.10965834, -0.01708995,  0.0339236 , ...,  0.05659021,
         0.06108881,  0.02438835],
       [ 0.0251045 ,  0.03409597,  0.05710061, ...,  0.08061729,
         0.07066568, -0.03364325],
       [ 0.00047341, -0.03664262,  0.0936758 , ..., -0.00429336,
        -0.02715965, -0.09085602]], shape=(7, 384), dtype=float32)

Training a Rebuilder model to clean our noisy sentences. Using same sentences as input/output

In [5]:
rebuilder = MLPRegressor(hidden_layer_sizes=(256, 128), max_iter=1000, random_state=42)
rebuilder.fit(X, X)

print("Model trained successfully.")

rebuilder

Model trained successfully.


,"loss loss: {'squared_error', 'poisson'}, default='squared_error'The loss function to use when training the weights. Note that the""squared error"" and ""poisson"" losses actually implement""half squares error"" and ""half poisson deviance"" to simplify thecomputation of the gradient. Furthermore, the ""poisson"" loss internally usesa log-link (exponential as the output activation function) and requires``y >= 0``... versionchanged:: 1.7 Added parameter `loss` and option 'poisson'.",'squared_error'
,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(256, ...)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the regressor will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate ``learning_rate_`` at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when solver='sgd'.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",1000
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True


### Using K-Nearest Neighbors for Sentence Reconstruction

Instead of generating text character-by-character, we use a `NearestNeighbors` model to map the clean embeddings back to our 'library' of known clean sentences.

In [6]:
knn = NearestNeighbors(n_neighbors=1, metric='cosine')
knn.fit(X)

knn

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",1
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


Adding Noise to embeddings in order to simulate communication over a noisy channel like a physical medium.

For generating noise, we're using Gaussian noise so it closely resembles real world.

In [7]:
def test_reconstruction_with_knn(noise_level):
    # 1. Add Gaussian noise to the embeddings
    noise = np.random.normal(0, noise_level, X.shape)
    noisy_X = X + noise

    # 2. Clean noisy data using our MLP Rebuilder
    rebuilt_X = rebuilder.predict(noisy_X)

    # 3. Use KNN to convert our noisy embeddings back into text
    distances, indices = knn.kneighbors(rebuilt_X)

    results = []
    for i, match_idx in enumerate(indices):
        predicted_sentence = sentences[match_idx[0]]

        # Calculate similarities for the table
        noisy_sim = cosine_similarity([X[i]], [noisy_X[i]])[0][0]
        rebuilt_sim = cosine_similarity([X[i]], [rebuilt_X[i]])[0][0]

        results.append({
            "Original Sentence": sentences[i],
            "Rebuilt (KNN) Match": predicted_sentence,
            "Noisy Sim": round(float(noisy_sim), 4),
            "Rebuilt Sim": round(float(rebuilt_sim), 4)
        })

    return pd.DataFrame(results)

# Test the application
display(test_reconstruction_with_knn(noise_level=0.1))

,Original Sentence,Rebuilt (KNN) Match,Noisy Sim,Rebuilt Sim
0,The quick brown fox jumps over the lazy dog.,The quick brown fox jumps over the lazy dog.,0.4836,0.5292
1,Data science is an interdisciplinary field.,I love working with python and pandas.,0.5295,0.5480
2,Artificial intelligence is transforming the wo...,Artificial intelligence is transforming the wo...,0.4676,0.5195
3,I love working with python and pandas.,I love working with python and pandas.,0.4368,0.5567
4,Jupyter notebooks are great for prototyping.,Jupyter notebooks are great for prototyping.,0.5147,0.6153
5,Natural language processing allows computers t...,Natural language processing allows computers t...,0.4453,0.5531
6,Machine learning models require clean data for...,Machine learning models require clean data for...,0.4789,0.5606
